# Compatbilização das malhas de zonas origem-destino para a Região Metropolitana de São Paulo

In [1]:
from utils import *

In [2]:
BASENAME = 'Metro_OD' # Nome da compatibilização
location = 'OD_RMSP'

## 1. Configuração das malhas

In [5]:
# Criação das malhas
od_id = {
    1987: 'Zona87',
    1997: 'Zona97',
    2007: 'Zona07',
    2017: 'NumeroZona',
    2023: 'NumeroZona',
}

malhas = {}
for k, v in od_id.items():
    malhas[k] = makeCensusGrid('metrosp/Zonas OD RMSP.gpkg',
                                id_column='GEOID', 
                                layer=f'{k}',
                                len_higher_hierarchy=2,
                                geosys='SIRGAS2000')
    malhas[k].to_parquet(f'malhas/{BASENAME}_{k}.parquet')

## 2. Compatibilização recursiva

In [8]:
list_malhas = [(k,v) for k, v in malhas.items()]

def compatRoutine(a, b):
    n1, m1 = a
    n2, m2 = b
    print(f'Starting {n1}-{n2}...')
    G_compat = compatibility_graph(m1, m2)
    G_compat.compatManutencao()
    G_compat.compatDivisao(threshold=0.8)

    # Aplica séries de buffers
    for b in [-100, -50, 0]:
        G_compat.compatSobreposicao(buffer=b)
    # Sobreposição de isolados restantes
    #G_compat.compatSobreposicao(buffer=0, use_all=True)
    # Export
    G_compat.exportCompatFiles(BASENAME, n1, n2)
    # Make coverage from export
    m12 = makeCensusGrid(f'malhas/{BASENAME}_AMC.gpkg',
                            layer=f'{n1}-{n2}',
                            id_column='CD_PERIMETRO',
                            len_higher_hierarchy=2,
                            is_utm=True)
    print(f'{n1}-{n2} ok!')
    return (f'{n1}-{n2}', m12)

def recursiveCompat(coverage_list):
    if len(coverage_list)==2:
        m = compatRoutine(coverage_list[0], coverage_list[1])
        return m
    else:
        m = compatRoutine(coverage_list[0], recursiveCompat(coverage_list[1:]))
        return m

m = recursiveCompat(list_malhas)

Starting 2017-2023...


c:\Users\Pedro\anaconda3\envs\geo\Lib\site-packages\geopandas\tools\overlay.py:358: UserWarning: `keep_geom_type=True` in overlay resulted in 858 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  result = _collection_extract(result, geom_type, keep_geom_type_warning)


2017-2023 ok!
Starting 2007-2017-2023...


c:\Users\Pedro\anaconda3\envs\geo\Lib\site-packages\geopandas\tools\overlay.py:358: UserWarning: `keep_geom_type=True` in overlay resulted in 3110 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  result = _collection_extract(result, geom_type, keep_geom_type_warning)


2007-2017-2023 ok!
Starting 1997-2007-2017-2023...


c:\Users\Pedro\anaconda3\envs\geo\Lib\site-packages\geopandas\tools\overlay.py:358: UserWarning: `keep_geom_type=True` in overlay resulted in 1992 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  result = _collection_extract(result, geom_type, keep_geom_type_warning)


1997-2007-2017-2023 ok!
Starting 1987-1997-2007-2017-2023...


c:\Users\Pedro\anaconda3\envs\geo\Lib\site-packages\geopandas\tools\overlay.py:358: UserWarning: `keep_geom_type=True` in overlay resulted in 1066 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  result = _collection_extract(result, geom_type, keep_geom_type_warning)
c:\Users\Pedro\anaconda3\envs\geo\Lib\site-packages\geopandas\tools\overlay.py:358: UserWarning: `keep_geom_type=True` in overlay resulted in 22 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  result = _collection_extract(result, geom_type, keep_geom_type_warning)


1987-1997-2007-2017-2023 ok!
